In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
import subprocess
import sys

def install_if_missing(package, import_name=None):
    """Install package if not already available."""
    try:
        __import__(import_name or package)
        print(f"✅ {package} already installed")
    except ImportError:
        print(f"📦 Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        print(f"✅ {package} installed successfully")

# Core dependencies
install_if_missing("transformers")
install_if_missing("peft")
install_if_missing("accelerate")
install_if_missing("datasets")
install_if_missing("trl")
install_if_missing("tqdm")
install_if_missing("wandb")

print("\n" + "=" * 70)
print("✅ All packages ready!")
print("=" * 70)

# Verify GPU
import torch
print(f"\n GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM Total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"   CUDA Version: {torch.version.cuda}")

In [ ]:
!pip install -U math-verify

In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
import torch
from huggingface_hub import list_repo_files, hf_hub_download
from datetime import datetime
from peft import PeftModel, get_peft_model, prepare_model_for_kbit_training, LoraConfig, TaskType
from huggingface_hub import snapshot_download
import wandb
import os, re, json, shutil
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, PeftModel, TaskType, prepare_model_for_kbit_training
from huggingface_hub import HfApi, login as hf_login
from bitsandbytes.optim import AdamW8bit
import random
from math_verify import LatexExtractionConfig, parse, verify
import signal
from math_verify.parser import LatexExtractionConfig, ExprExtractionConfig

In [ ]:
# ========================= CONFIG =========================
STUDENT_ID = "meta-llama/Llama-3.2-1B-Instruct"
TEACHER_ID = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"

SEED = 42
GROUP_SIZE = 10
PROMPTS_PER_STEP = 2
MAX_NEW_TOKENS = 512
MAX_STEPS = 100
TEMPERATURE = 1.2
TOP_P = 0.99
LR = 1e-4
CLIP_EPS = 0.2
CHUNK_SIZE = 1 
ACCUMULATION_STEPS = 4
GRADE_TIMEOUT_SECONDS = 5
#MAX_TOTAL_LEN = 512

In [ ]:
# ---- LoRA ----
LORA_R = 64
LORA_ALPHA = 128
LORA_TARGET = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
LORA_DROPOUT = 0.0

In [ ]:
# ---- Checkpointing ----
CKPT_DIR = "./distilled_rl_checkpoint"
BEST_DIR = os.path.join(CKPT_DIR, "best")
LOG_FILE = "./distilled_rl_training_log.jsonl"
CKPT_EVERY = 20
VAL_EVERY = 20
VAL_BATCH_SIZE = 40
MAX_VAL_SAMPLES = 200

USE_WANDB = True
PUSH_TO_HUB = True
HF_HUB_ID = None

os.makedirs(CKPT_DIR, exist_ok=True)

In [ ]:
# ========================= AUTH =========================
HF_TOKEN = None
WANDB_API_KEY = None
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    try: HF_TOKEN = _secrets.get_secret("HF_TOKEN")
    except: pass
    try: WANDB_API_KEY = _secrets.get_secret("WANDB_API_KEY")
    except: pass
except:
    pass

if not HF_TOKEN: HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN: raise RuntimeError("HF_TOKEN missing.")
if not WANDB_API_KEY: WANDB_API_KEY = os.environ.get("WANDB_API_KEY")
if USE_WANDB and WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
else:
    USE_WANDB = False
    print("W&B disabled.")

hf_login(token=HF_TOKEN)
torch.manual_seed(SEED)

In [ ]:
# ========================= HELPERS =========================
def find_latest_checkpoint():
    """Check local first, then download from HF Hub if needed."""
    # 1. Scan local directory first
    if os.path.exists(CKPT_DIR):
        ckpts = []
        for d in os.listdir(CKPT_DIR):
            if d.startswith("step"):
                try:
                    num = int(d.replace("step", ""))
                    if os.path.exists(os.path.join(CKPT_DIR, d, "training_state.json")):
                        ckpts.append((num, d))
                except:
                    continue
        if ckpts:
            latest_num, latest_name = sorted(ckpts)[-1]
            return os.path.join(CKPT_DIR, latest_name), latest_num

    # 2. No local checkpoint found -> look on HF Hub
    print(f"  No local checkpoint found. Scanning HF Hub: {HF_HUB_ID}")
    try:
        # List all files in the repo
        files = list_repo_files(HF_HUB_ID, token=HF_TOKEN)
        steps = []
        for f in files:
            match = re.match(r"step(\d+)/", f)
            if match:
                steps.append(int(match.group(1)))

        if not steps:
            print("  No checkpoints found on HF Hub.")
            return None, 0

        latest_step = max(steps)
        print(f"  Found latest checkpoint on HF: step{latest_step}. Downloading...")

        # Download all files for this step
        local_path = os.path.join(CKPT_DIR, f"step{latest_step}")
        os.makedirs(local_path, exist_ok=True)

        files_to_download = [
            "adapter_config.json",
            "adapter_model.safetensors",
            "optimizer.pt",
            "training_state.json",
            "rng.pt",
            "training_log.jsonl"
        ]

        for fname in files_to_download:
            try:
                hf_hub_download(
                    repo_id=HF_HUB_ID,
                    filename=f"step{latest_step}/{fname}",
                    local_dir=CKPT_DIR,
                    token=HF_TOKEN,
                )
            except Exception as e:
                # If safetensors doesn't exist, try .bin
                if fname == "adapter_model.safetensors":
                    try:
                        hf_hub_download(
                            repo_id=HF_HUB_ID,
                            filename=f"step{latest_step}/adapter_model.bin",
                            local_dir=CKPT_DIR,
                            token=HF_TOKEN,
                        )
                    except:
                        print(f"  Warning: Could not download adapter_model (tried .safetensors and .bin)")
                else:
                    # training_log.jsonl might not exist in older checkpoints
                    if fname != "training_log.jsonl":
                        print(f"  Warning: Could not download {fname}")

        print(f"  Successfully downloaded step{latest_step} to {local_path}")
        return local_path, latest_step

    except Exception as e:
        print(f"  Error downloading from HF: {e}")
        return None, 0

def split_reasoning(text):
    if "<think" in text and "</think" in text:
        open_tag = re.search(r"<think[^>]*>", text)
        close_tag = re.search(r"</think\s*>", text)
        if open_tag and close_tag and close_tag.start() > open_tag.end():
            return text[open_tag.end():close_tag.start()].strip(), text[close_tag.end():].strip()
    return None, text



def extract_boxed_content(text):
    idx = text.rfind("\\boxed")
    if idx == -1:
        return None
    i = text.find("{", idx)
    if i == -1:
        return None
    depth = 0
    for j in range(i, len(text)):
        if text[j] == "{":
            depth += 1
        elif text[j] == "}":
            depth -= 1
            if depth == 0:
                return text[i + 1:j]
    return None

def parse_number(s):
    if s is None: return None
    s = s.replace(",", "").replace("$", "").strip()
    m = re.findall(r"-?\d+\.?\d*/\d+\.?\d*|-?\d+\.?\d*", s)
    if not m: return None
    tok = m[-1]
    try:
        if "/" in tok:
            num, den = tok.split("/")
            return float(num) / float(den)
        return float(tok)
    except: return None

def extract_answer(text):
    reasoning, answer_region = split_reasoning(text)
    if reasoning is None:
        answer_region = text
    boxed = extract_boxed_content(answer_region)
    if boxed is not None:
        return parse_number(boxed)
    return None  # no boxed answer -> no credit, period

class _GradeTimeout(Exception):
    pass


def _grade_timeout_handler(signum, frame):
    raise _GradeTimeout()


def extract_answer_regex(text):
    """
    Extract a numerical answer from plain text using regex fallback.
    Looks for explicit "answer is ...", "therefore ...", and then last numbers.
    """
    if not text:
        return None

    # 1. Explicit answer patterns
    ANSWER_PATTERNS = [
        r"(?:final\s+answer)\s*(?:is|=|:)\s*([^\n.]+)",
        r"(?:the\s+answer)\s*(?:is|=|:)\s*([^\n.]+)",
        r"(?:therefore|thus|so)\s*,?\s*(?:the\s+)?answer\s*(?:is|=|:)\s*([^\n.]+)",
    ]
    for pat in ANSWER_PATTERNS:
        matches = re.findall(pat, text, flags=re.IGNORECASE)
        if matches:
            val = parse_number(matches[-1])
            if val is not None:
                return val

    # 2. Last number in the last sentence
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    if sentences:
        val = parse_number(sentences[-1])
        if val is not None:
            return val

    # 3. Last number anywhere
    nums = re.findall(r"-?\d[\d,]*\.?\d*", text)
    if nums:
        val = parse_number(nums[-1])
        if val is not None:
            return val

    return None

def compute_reward(completion, gt_answer):
    """
    Binary reward. Tries math_verify first; only falls back to the regex
    extractor when math_verify itself couldn't produce a definitive parse.
    """
    signal.signal(signal.SIGALRM, _grade_timeout_handler)
    signal.alarm(GRADE_TIMEOUT_SECONDS)
    try:
        pred_expr = parse(
            completion,
            extraction_config=[ExprExtractionConfig(), LatexExtractionConfig()],
        )
        gt_wrapped = (
            gt_answer if ("\\boxed" in gt_answer or "$" in gt_answer)
            else f"\\boxed{{{gt_answer}}}"
        )
        gt_expr = parse(
            gt_wrapped,
            extraction_config=[ExprExtractionConfig(), LatexExtractionConfig()],
        )
        if pred_expr and gt_expr:
            return 1.0 if verify(gt_expr, pred_expr) else 0.0
    except _GradeTimeout:
        print(f"  [compute_reward] TIMEOUT after {GRADE_TIMEOUT_SECONDS}s")
        return 0.0
    except Exception:
        pass  # math_verify couldn't parse this one - fall through to regex
    finally:
        signal.alarm(0)

    # Regex fallback (plain numbers, explicit "answer is ..."), only reached
    # when math_verify didn't produce a definitive parsed verdict above.
    pred_num = extract_answer_regex(completion)
    gt_num = parse_number(gt_answer)
    if pred_num is not None and gt_num is not None:
        return 1.0 if abs(pred_num - gt_num) < 1e-4 else 0.0

    return 0.0


def _math_verify_parseable(s):
    """Lightweight parseability check for data-loading filters, using
    math_verify itself rather than the old numeric-only parse_number - this
    accepts fractions/radicals/algebraic answers that parse_number would
    have wrongly rejected (or worse, silently mis-extracted a stray digit
    from), while still rejecting genuinely unparseable garbage."""
    try:
        wrapped = s if ("\\boxed" in s or "$" in s) else f"\\boxed{{{s}}}"
        return len(parse(wrapped, extraction_config=[LatexExtractionConfig(boxed_match_priority=0)])) > 0
    except Exception:
        return False

# ========================= TOKENIZER =========================
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(STUDENT_ID, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# ========================= DATASET =========================
# ========================= V8 DATA POOL (MATH + mAceReason + GSM8K) =========================
print("Loading V8 data pool...")


def load_math_v8(limit=5000):
    print("  Loading MATH (qwedsacf/competition_math) Levels 3-5...")
    try:
        ds = load_dataset("qwedsacf/competition_math", split="train", token=HF_TOKEN)
    except:
        ds = load_dataset("hendrycks/competition_math", split="train", token=HF_TOKEN)

    def level_filter(ex):
        lv = ex['level']
        if isinstance(lv, str):
            match = re.search(r'(\d+)', lv)
            return int(match.group(1)) >= 3 if match else False
        return int(lv) >= 3

    def solution_length_filter(ex):
        return len(ex["solution"]) < 800

    ds = ds.filter(level_filter)
    ds = ds.filter(solution_length_filter)
    ds = ds.filter(lambda ex: len(ex["problem"]) < 800)   # add this
    ds = ds.shuffle(seed=42).select(range(min(limit, len(ds))))

    data = []
    for ex in ds:
        gt_raw = extract_boxed_content(ex["solution"])
        if gt_raw is None:
            continue

        if not _math_verify_parseable(gt_raw):
            continue
        data.append({
            "question": ex["problem"],
            "answer": gt_raw,  # raw LaTeX, e.g. "\\frac{7}{12}" - not a flattened float
            "source": "MATH"
        })
    print(f"    Loaded {len(data)} MATH problems (Levels 3-5).")
    return data


def load_acereason_v8(limit=3000):
    print("  Loading AceReason-Math...")
    ds = load_dataset("nvidia/AceReason-Math", split="train", token=HF_TOKEN)
    # NOTE: verify the exact field names yourself with a quick print(ds[0]) -

    ds = ds.shuffle(seed=42).select(range(min(limit, len(ds))))
    data = []
    for ex in ds:
        q = ex.get("problem", ex.get("question", ""))
        raw_gt = ex.get("answer", ex.get("solution", ""))
        if not q or not raw_gt:
            continue
        raw_gt = str(raw_gt)

        if not _math_verify_parseable(raw_gt):
            continue
        data.append({
            "question": q,
            "answer": raw_gt,
            "source": "AceReason"
        })
    print(f"    Loaded {len(data)} AceReason problems.")
    return data


def load_gsm8k_v8(limit=6000):
    print("  Loading GSM8K (anchor)...")
    ds = load_dataset("openai/gsm8k", "main", split="train", token=HF_TOKEN)
    ds = ds.shuffle(seed=42).select(range(min(limit, len(ds))))
    data = []
    for ex in ds:
        gt = ex["answer"].split("####")[-1].replace(",", "").strip()
        if parse_number(gt) is None:  
            continue
        data.append({
            "question": ex["question"],
            "answer": gt,
            "source": "GSM8K"
        })
    print(f"    Loaded {len(data)} GSM8K problems.")
    return data


v8_pool = []
#v8_pool.extend(load_math_v8(5000))
#v8_pool.extend(load_acereason_v8(2000))
v8_pool.extend(load_gsm8k_v8(6000))

print(f"Total V8 pool size: {len(v8_pool)} problems")

full_train = load_dataset("openai/gsm8k", "main", split="train", token=HF_TOKEN)
split = full_train.train_test_split(test_size=0.1, seed=42)
train_ds = split["train"]
val_ds = split["test"]


# ========================= MODELS =========================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,    
    bnb_4bit_use_double_quant=True,
)

# --- Student (cuda:0) ---
print("Loading student on cuda:0...")
student_base = AutoModelForCausalLM.from_pretrained(
    STUDENT_ID, quantization_config=bnb_config, device_map={"": "cuda:0"},
    torch_dtype=torch.bfloat16, trust_remote_code=True,
)
student_base = prepare_model_for_kbit_training(student_base)


latest_ckpt, start_step = find_latest_checkpoint()
best_val_score = 0.0
best_step = 0
wandb_run_id = None
base_val_score = None
state = None

if latest_ckpt:
    print(f"Resuming LoRA from: {latest_ckpt}")
    student = PeftModel.from_pretrained(student_base, latest_ckpt, is_trainable=True)
    state_path = os.path.join(latest_ckpt, "training_state.json")
    if os.path.exists(state_path):
        with open(state_path, "r") as f:
            state = json.load(f)
        start_step = state.get("step", 0)
        best_val_score = state.get("best_val_score", 0.0)
        best_step = state.get("best_step", 0)
        wandb_run_id = state.get("wandb_run_id", None)
        base_val_score = state.get("base_val_score", None)
        print(f"  Resumed step={start_step}, best_val={best_val_score:.4f}")
else:
    print("Starting fresh LoRA...")
    lora_config = LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, target_modules=LORA_TARGET,
        lora_dropout=LORA_DROPOUT, bias="none", task_type=TaskType.CAUSAL_LM,
    )
    student = get_peft_model(student_base, lora_config)

student.print_trainable_parameters()
student.enable_input_require_grads()
student.gradient_checkpointing_enable()
student.config.use_cache = False

#optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, student.parameters()), lr=LR)
optimizer = AdamW8bit(filter(lambda p: p.requires_grad, student.parameters()), lr=LR)
if latest_ckpt:
    opt_path = os.path.join(latest_ckpt, "optimizer.pt")
    if os.path.exists(opt_path):
        optimizer.load_state_dict(torch.load(opt_path, map_location="cuda:0"))




# --- Teacher (cuda:1, 4-bit) ---
print("Loading teacher on cuda:1...")
teacher = AutoModelForCausalLM.from_pretrained(
    TEACHER_ID,
    quantization_config=bnb_config,            
    device_map={"": "cuda:1"},
    torch_dtype=torch.float16,                 
    trust_remote_code=True,
)
teacher.eval()
for p in teacher.parameters(): p.requires_grad = False

print(f"GPU0: {torch.cuda.memory_allocated('cuda:0')/1e9:.2f}GB, GPU1: {torch.cuda.memory_allocated('cuda:1')/1e9:.2f}GB")


# ========================= CORE FUNCTIONS =========================
def get_token_logps(model, input_ids, attention_mask, prompt_lens, return_entropy=False, device=None):
    """
    Compute token log-probs. If device is specified, move tensors there first.
    This allows running teacher on GPU 1 while student stays on GPU 0.
    """
    if device is not None:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        prompt_lens = prompt_lens.to(device)
    else:
        # Use the model's device (backward compatible)
        device = next(model.parameters()).device
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        prompt_lens = prompt_lens.to(device)

    all_token_logps, all_masks = [], []
    entropy_sum, entropy_count = 0.0, 0

    for i in range(0, input_ids.shape[0], CHUNK_SIZE):
        chunk_ids = input_ids[i:i+CHUNK_SIZE]
        chunk_mask = attention_mask[i:i+CHUNK_SIZE]
        chunk_prompts = prompt_lens[i:i+CHUNK_SIZE]

        out = model(input_ids=chunk_ids, attention_mask=chunk_mask)
        logits = out.logits[:, :-1, :]
        targets = chunk_ids[:, 1:]

        target_logits = torch.gather(logits, 2, targets.unsqueeze(-1)).squeeze(-1)
        logsumexp = torch.logsumexp(logits, dim=-1)
        token_logps = target_logits - logsumexp

        seq_len = chunk_ids.shape[1] - 1
        pos = torch.arange(seq_len, device=device).unsqueeze(0)
        comp_mask = (pos >= (chunk_prompts.unsqueeze(1) - 1)).long() * chunk_mask[:, 1:]

        if return_entropy:
            probs = torch.softmax(logits, dim=-1)
            token_entropy = -(probs * torch.log(probs + 1e-10)).sum(dim=-1)
            entropy_sum += (token_entropy * comp_mask).sum().item()
            entropy_count += comp_mask.sum().item()

        all_token_logps.append(token_logps)
        all_masks.append(comp_mask)
        del out, logits

    token_logps = torch.cat(all_token_logps, dim=0).float()
    comp_mask = torch.cat(all_masks, dim=0)

    if return_entropy:
        avg_entropy = entropy_sum / max(entropy_count, 1)
        return token_logps, comp_mask, avg_entropy
    return token_logps, comp_mask

def build_prompt(question):
    user_content = (
        "Please reason step by step, and put your final answer within \\boxed{}.\n\n"
        f"{question}"
    )
    chat = [{"role": "user", "content": user_content}]
    return tokenizer.apply_chat_template(chat, add_generation_prompt=True, tokenize=False)


# ========================= DYNAMIC MAX_TOTAL_LEN =========================
print("Measuring prompt length distribution across the V8 pool...")
_prompt_token_lens = [
    len(tokenizer(build_prompt(ex["question"]), add_special_tokens=False).input_ids)
    for ex in v8_pool
]
_prompt_token_lens.sort()
_p99_prompt_len = _prompt_token_lens[int(0.99 * (len(_prompt_token_lens) - 1))]
MAX_TOTAL_LEN = _p99_prompt_len + MAX_NEW_TOKENS
print(f"  Prompt lengths (tokens): min={_prompt_token_lens[0]}, "
      f"median={_prompt_token_lens[len(_prompt_token_lens)//2]}, "
      f"p99={_p99_prompt_len}, max={_prompt_token_lens[-1]}")
print(f"  MAX_TOTAL_LEN set to {MAX_TOTAL_LEN} (p99 prompt length + MAX_NEW_TOKENS; "
      f"was a fixed 512 guess)")


def collect_rollouts(examples):
    rows, prompt_lens_list, rewards, group_ids, sources = [], [], [], [], []  
    box_hits = 0
    student.eval()

    for gi, ex in enumerate(examples):
        prompt_text = build_prompt(ex["question"])
        prompt_ids = tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False).input_ids.to("cuda:0")
        plen = prompt_ids.shape[1]

        with torch.no_grad():
            gen_inputs = prompt_ids.repeat(GROUP_SIZE, 1)
            gen_attn = torch.ones_like(gen_inputs)
            generated = student.generate(
                gen_inputs, attention_mask=gen_attn, max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True, temperature=TEMPERATURE, top_p=TOP_P,
                pad_token_id=tokenizer.pad_token_id,
            )

        raw_answer = ex["answer"]
        if ex.get("source") == "GSM8K":
            gt_answer = raw_answer.split("####")[-1].replace(",", "").strip()
        else:
            gt_answer = raw_answer.strip()

        for row in generated:
            truncated_row = row[:MAX_TOTAL_LEN]
            rows.append(truncated_row.clone())  # row is a 1D tensor on cuda:0
            prompt_lens_list.append(plen)
            text = tokenizer.decode(truncated_row[plen:], skip_special_tokens=True)
            rewards.append(compute_reward(text, gt_answer))
            group_ids.append(gi)
            sources.append(ex.get("source", "unknown"))       
            if extract_boxed_content(text) is not None:
                box_hits += 1

        del generated, gen_inputs
        torch.cuda.empty_cache()

    box_rate = box_hits / max(len(rows), 1)

    # Return raw rows (unpadded), prompt lengths, rewards, group ids, box rate
    return rows, prompt_lens_list, rewards, group_ids, sources, box_rate 

def group_normalize(rewards, group_ids):
    advantages = torch.zeros_like(rewards)
    for g in group_ids.unique():
        mask = group_ids == g
        r = rewards[mask]
        std = r.std()
        if std > 0:
            advantages[mask] = (r - r.mean()) / (std + 1e-4)
    return advantages

# ========================= DYNAMIC SAMPLING HELPER =========================
def is_informative_group(rewards, group_ids):
    """Check if ANY group has mixed rewards (not all 0 and not all 1)."""
    for g in group_ids.unique():
        mask = group_ids == g
        r = rewards[mask]
        if r.std() > 0:
            return True
    return False


def get_informative_group(max_attempts=10):
    for attempt in range(max_attempts):
        indices = torch.randint(0, len(v8_pool), (PROMPTS_PER_STEP,))
        examples = [v8_pool[int(i)] for i in indices]

        student.eval()
        rows, prompt_lens_list, rewards, group_ids, sources, box_rate = collect_rollouts(examples)  

        rewards_tensor = torch.tensor(rewards, dtype=torch.float32, device="cuda:0")
        group_ids_tensor = torch.tensor(group_ids, device="cuda:0")

        if is_informative_group(rewards_tensor, group_ids_tensor):
            return rows, prompt_lens_list, rewards_tensor, group_ids_tensor, sources, box_rate  

        del rows, prompt_lens_list, rewards, group_ids
        torch.cuda.empty_cache()

    return None
# ========================= VALIDATION (FIXED: returns tuple) =========================
@torch.no_grad()
def run_validation(model, dataset, batch_size=VAL_BATCH_SIZE, max_samples=MAX_VAL_SAMPLES):
    model.eval()
    ds = dataset.select(range(min(max_samples, len(dataset))))
    tokenizer.padding_side = "left"

    correct, total = 0, 0
    predictions = []

    for i in range(0, len(ds), batch_size):
        batch = ds[i:i+batch_size]
        prompts_text = [build_prompt(q) for q in batch["question"]]
        gts = [a.split("####")[-1].replace(",", "").strip() for a in batch["answer"]]

 
        inputs = tokenizer(prompts_text, return_tensors="pt", padding=True, truncation=True,
                            max_length=512, add_special_tokens=False)
        inputs = {k: v.to("cuda:0") for k, v in inputs.items()}
        padded_len = inputs["input_ids"].shape[1]

        outputs = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=tokenizer.pad_token_id)

        for j, out in enumerate(outputs):
            resp = tokenizer.decode(out[padded_len:], skip_special_tokens=True)
            reward = compute_reward(resp, gts[j])
            correct += reward
            total += 1
            predictions.append({"gt": gts[j], "pred": extract_answer(resp), "correct": bool(reward)})

    tokenizer.padding_side = "right"
    acc = correct / total if total > 0 else 0.0
    return acc, predictions

# ========================= W&B INIT =========================
if USE_WANDB:
    run = wandb.init(
        project="llama-distilled-rl",
        id=wandb_run_id, resume="allow" if wandb_run_id else None,
        config={
            "student": STUDENT_ID, "teacher": TEACHER_ID, "group_size": GROUP_SIZE,
            "lr": LR, "clip_eps": CLIP_EPS, "method": "DistilledRL_Pure",
        },
    )
    if wandb_run_id is None:
        wandb_run_id = run.id
    print(f"W&B RUN ID: {run.id}")
    print(f"W&B URL: {run.url}")

# ========================= BASELINE EVAL =========================
if base_val_score is None and start_step == 0:
    base_val_score = None
    print(f"\n=== Using pre-computed base student Pass@1: {base_val_score:.4f} ===\n")
    if USE_WANDB:
        wandb.run.summary["base_val_pass@1"] = base_val_score


# ========================= TRAINING LOOP ========================

# ========================= HYPERPARAMETERS =========================
# ACCUMULATION_STEPS = number of fresh rollout batches
# CHUNK_SIZE = how many responses per chunk (adjust to fit GPU memory)
# Effective batch = ACCUMULATION_STEPS * PROMPTS_PER_STEP * GROUP_SIZE


# ========================= HELPER: move tensors to CPU =========================
def to_cpu(tensor):
    return tensor.detach().cpu() if tensor is not None else None

# ========================= TRAINING LOOP =========================
TRAIN_MICROBATCH_ROWS = 2
print(f"Starting V10 Distilled RL from step {start_step}...")
n_dead_steps = 0

for step in range(start_step, MAX_STEPS):
    torch.cuda.empty_cache()
    student.train()
    torch.cuda.reset_peak_memory_stats()

    optimizer.zero_grad()
    accumulated_loss = 0.0
    grad_norm = 0.0
    post_clip_frac = 0.0

    all_frozen_chunks = []
    # ---- NEW: per-source reward accumulators ----
    source_reward_sum = {}
    source_reward_count = {}

    try:
        for acc_step in range(ACCUMULATION_STEPS):
            result = get_informative_group(max_attempts=5)
            if result is None:
                print(f"Step {step:03d} | Accum {acc_step+1}/{ACCUMULATION_STEPS} | No group, skipping.")
                continue

            rows, prompt_lens_list, rewards, group_ids, sources, box_rate = result  # + sources

            # ---- NEW: accumulate per-source rewards ----
            for src, rew in zip(sources, rewards.tolist()):
                source_reward_sum[src] = source_reward_sum.get(src, 0.0) + rew
                source_reward_count[src] = source_reward_count.get(src, 0) + 1

            advantages = group_normalize(rewards, group_ids)
            if advantages.abs().max().item() == 0:
                print(f"Step {step:03d} | Accum {acc_step+1}/{ACCUMULATION_STEPS} | Dead, skipping.")
                del rows, prompt_lens_list, rewards, group_ids, advantages
                torch.cuda.empty_cache()
                continue

            n_total = len(rows)
            chunk_size = min(TRAIN_MICROBATCH_ROWS, n_total)
            prompt_lens_tensor = torch.tensor(prompt_lens_list, device="cuda:0")

            for c in range(0, n_total, chunk_size):
                sl = slice(c, min(c + chunk_size, n_total))
                chunk_rows = rows[sl]
                chunk_prompt_lens = prompt_lens_tensor[sl]

                local_max = max(r.shape[0] for r in chunk_rows)
                chunk_input_ids = torch.full((len(chunk_rows), local_max), tokenizer.pad_token_id,
                                             dtype=torch.long, device="cuda:0")
                chunk_attention_mask = torch.zeros((len(chunk_rows), local_max), dtype=torch.long, device="cuda:0")
                for i, row in enumerate(chunk_rows):
                    length = row.shape[0]
                    chunk_input_ids[i, :length] = row
                    chunk_attention_mask[i, :length] = 1

                A_chunk = advantages[sl].unsqueeze(1)

                with torch.no_grad():
                    old_logps_chunk, comp_mask_chunk = get_token_logps(
                        student, chunk_input_ids, chunk_attention_mask, chunk_prompt_lens
                    )
                    old_logps_chunk = old_logps_chunk.cpu()
                    comp_mask_chunk = comp_mask_chunk.cpu()
                    A_chunk_cpu = A_chunk.cpu()

                    has_positive_adv = bool((A_chunk_cpu > 0).any().item())
                    if has_positive_adv:
                        teach_logps_chunk, _ = get_token_logps(
                            teacher, chunk_input_ids, chunk_attention_mask, chunk_prompt_lens,
                            device="cuda:1"
                        )
                        teach_logps_chunk = teach_logps_chunk.cpu()
                        log_rho = teach_logps_chunk - old_logps_chunk
                        rho = torch.exp(log_rho)
                        rho_bar = torch.clamp(rho, 1.0 / 3.0, 3.0)
                        log_rho_bar = torch.log(rho_bar + 1e-10)
                        sum_log_rho = (log_rho_bar * comp_mask_chunk).sum(dim=1, keepdim=True)
                        count = comp_mask_chunk.sum(dim=1, keepdim=True).clamp(min=1)
                        geo_mean = torch.exp(sum_log_rho / count)
                        rho_tilde = rho_bar / (geo_mean + 1e-10)
                        w_chunk = torch.where(A_chunk_cpu > 0, rho_tilde, torch.ones_like(rho_tilde))
                    else:
                        rho_bar = torch.ones_like(old_logps_chunk)
                        rho_tilde = torch.ones_like(old_logps_chunk)
                        w_chunk = torch.ones_like(old_logps_chunk)

                    frozen = {
                        "input_ids": chunk_input_ids.cpu(),
                        "attention_mask": chunk_attention_mask.cpu(),
                        "prompt_lens": chunk_prompt_lens.cpu(),
                        "A": A_chunk_cpu,
                        "old_logps": old_logps_chunk,
                        "comp_mask": comp_mask_chunk,
                        "w": w_chunk,
                        "rho_bar": rho_bar,
                        "rho_tilde": rho_tilde,
                        "rewards": rewards[sl].cpu(),
                        "adv_mean": A_chunk.mean().item(),
                        "adv_std": A_chunk.std().item(),
                        "pct_neg": (A_chunk <= 0).float().mean().item(),
                    }
                    _, _, avg_entropy = get_token_logps(
                        student, chunk_input_ids, chunk_attention_mask, chunk_prompt_lens,
                        return_entropy=True
                    )
                    resp_lengths = (chunk_input_ids != tokenizer.pad_token_id).sum(dim=1) - chunk_prompt_lens
                    avg_len = resp_lengths.float().mean().item()
                    mean_rho_bar = (rho_bar * comp_mask_chunk).sum() / comp_mask_chunk.sum().clamp(min=1)
                    mean_rho_tilde = (rho_tilde * comp_mask_chunk).sum() / comp_mask_chunk.sum().clamp(min=1)
                    frozen["avg_entropy"] = avg_entropy
                    frozen["avg_len"] = avg_len
                    frozen["mean_rho_bar"] = mean_rho_bar.item()
                    frozen["mean_rho_tilde"] = mean_rho_tilde.item()
                    frozen["box_rate"] = box_rate
                    all_frozen_chunks.append(frozen)

                del chunk_input_ids, chunk_attention_mask, chunk_prompt_lens, A_chunk
                del old_logps_chunk, comp_mask_chunk
                torch.cuda.empty_cache()

            del rows, prompt_lens_list, rewards, group_ids, advantages
            torch.cuda.empty_cache()

        if len(all_frozen_chunks) == 0:
            print(f"Step {step:03d} | No chunks collected, skipping.")
            del all_frozen_chunks
            torch.cuda.empty_cache()
            continue

        total_valid_chunks = len(all_frozen_chunks)

        for frozen in all_frozen_chunks:
            chunk_input_ids = frozen["input_ids"].to("cuda:0")
            chunk_attention_mask = frozen["attention_mask"].to("cuda:0")
            chunk_prompt_lens = frozen["prompt_lens"].to("cuda:0")
            chunk_old_logps = frozen["old_logps"].to("cuda:0")
            chunk_comp_mask = frozen["comp_mask"].to("cuda:0")
            chunk_w = frozen["w"].to("cuda:0")
            chunk_A = frozen["A"].to("cuda:0")

            new_logps_chunk, _ = get_token_logps(
                student, chunk_input_ids, chunk_attention_mask, chunk_prompt_lens
            )
            log_r = torch.clamp(new_logps_chunk - chunk_old_logps, -5.0, 5.0)
            r = torch.exp(log_r)
            r_clipped = torch.clamp(r, 1.0 - CLIP_EPS, 1.0 + CLIP_EPS)

            obj = torch.min(r * chunk_w * chunk_A, r_clipped * chunk_w * chunk_A)
            per_seq_obj = (obj * chunk_comp_mask).sum(dim=1) / chunk_comp_mask.sum(dim=1).clamp(min=1)
            loss = -per_seq_obj.mean()
            loss = loss / total_valid_chunks

            if torch.isnan(loss) or torch.isinf(loss):
                raise FloatingPointError(f"NaN/Inf loss at step {step}")

            loss.backward()
            accumulated_loss += loss.item() * total_valid_chunks

            del chunk_input_ids, chunk_attention_mask, chunk_prompt_lens, chunk_old_logps
            del chunk_comp_mask, chunk_w, chunk_A, new_logps_chunk, log_r, r, r_clipped, obj, loss
            torch.cuda.empty_cache()

    except (FloatingPointError, RuntimeError) as e:
        print(f"Step {step:03d} | {e} -> discarding entire update")
        optimizer.zero_grad(set_to_none=True)
        del all_frozen_chunks
        torch.cuda.empty_cache()
        continue

    has_nan_grad = False
    for p in student.parameters():
        if p.grad is not None and (torch.isnan(p.grad).any() or torch.isinf(p.grad).any()):
            has_nan_grad = True
            break
    if has_nan_grad:
        print(f"Step {step:03d} | NaN gradient - skipping update")
        optimizer.zero_grad(set_to_none=True)
        grad_norm = 0.0
        post_clip_frac = 0.0
    else:
        grad_norm = float(torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0))
        if grad_norm > 1000:
            print(f"Step {step:03d} | Spike: grad_norm={grad_norm:.1f} - skipping")
            optimizer.zero_grad(set_to_none=True)
            post_clip_frac = 0.0
        else:
            optimizer.step()

            post_clip_sum = 0.0
            post_count = 0
            with torch.no_grad():
                for frozen in all_frozen_chunks:
                    chunk_input_ids = frozen["input_ids"].to("cuda:0")
                    chunk_attention_mask = frozen["attention_mask"].to("cuda:0")
                    chunk_prompt_lens = frozen["prompt_lens"].to("cuda:0")
                    chunk_old_logps = frozen["old_logps"].to("cuda:0")
                    chunk_comp_mask = frozen["comp_mask"].to("cuda:0")

                    new_logps_post, _ = get_token_logps(
                        student, chunk_input_ids, chunk_attention_mask, chunk_prompt_lens
                    )
                    r_post = torch.exp(torch.clamp(new_logps_post - chunk_old_logps, -5.0, 5.0))
                    clip_mask_post = ((r_post > 1 + CLIP_EPS) | (r_post < 1 - CLIP_EPS)).float()
                    post_clip_sum += (clip_mask_post * chunk_comp_mask).sum().item()
                    post_count += chunk_comp_mask.sum().item()

                    del chunk_input_ids, chunk_attention_mask, chunk_prompt_lens, chunk_old_logps, chunk_comp_mask, new_logps_post
                    torch.cuda.empty_cache()
            post_clip_frac = post_clip_sum / post_count if post_count > 0 else 0.0

            optimizer.zero_grad(set_to_none=True)

    avg_loss = accumulated_loss / total_valid_chunks if total_valid_chunks > 0 else 0.0

    n = len(all_frozen_chunks)
    avg_reward = sum(f["rewards"].mean().item() for f in all_frozen_chunks) / n
    avg_adv_mean = sum(f["adv_mean"] for f in all_frozen_chunks) / n
    avg_adv_std = sum(f["adv_std"] for f in all_frozen_chunks) / n
    avg_pct_neg = sum(f["pct_neg"] for f in all_frozen_chunks) / n
    avg_entropy = sum(f["avg_entropy"] for f in all_frozen_chunks) / n
    avg_len = sum(f["avg_len"] for f in all_frozen_chunks) / n
    avg_rho_bar = sum(f["mean_rho_bar"] for f in all_frozen_chunks) / n
    avg_rho_tilde = sum(f["mean_rho_tilde"] for f in all_frozen_chunks) / n
    avg_box_rate = sum(f["box_rate"] for f in all_frozen_chunks) / n

    log_entry = {
        "step": step,
        "loss": avg_loss,
        "avg_reward": avg_reward,
        "adv_mean": avg_adv_mean,
        "adv_std": avg_adv_std,
        "rho_bar": avg_rho_bar,
        "rho_tilde": avg_rho_tilde,
        "pct_neg": avg_pct_neg,
        "resp_len": avg_len,
        "entropy": avg_entropy,
        "clip_frac": post_clip_frac,
        "grad_norm": grad_norm,
        "box_rate": avg_box_rate,
        "num_chunks": n,
    }
    # ---- NEW: per-source reward fields ----
    for src in source_reward_count:
        log_entry[f"reward_{src}"] = source_reward_sum[src] / source_reward_count[src]
        log_entry[f"n_{src}"] = source_reward_count[src]

    with open(LOG_FILE, "a") as f:
        f.write(json.dumps(log_entry) + "\n")
    if USE_WANDB:
        wandb.log({k: v for k, v in log_entry.items() if k != "step"}, step=step)

    per_source_str = " | ".join(
        f"{src}: rew={source_reward_sum[src]/source_reward_count[src]:.2f} (n={source_reward_count[src]})"
        for src in source_reward_count
    )
    print(f"Step {step:03d} | Loss: {avg_loss:.4f} | Rew: {avg_reward:.2f} | "
          f"Adv: {avg_adv_mean:.2f}±{avg_adv_std:.2f} | Len: {avg_len:.0f} | "
          f"box: {avg_box_rate:.3f} | Ent: {avg_entropy:.3f} | clip: {post_clip_frac:.4f} | "
          f"rho: {avg_rho_bar:.2f}->{avg_rho_tilde:.2f} | chunks: {n}")
    print(f"  Per-source: {per_source_str}")

    del all_frozen_chunks
    torch.cuda.empty_cache()


# ========================= CHECKPOINT =========================
    if (step + 1) % CKPT_EVERY == 0:
        ckpt_path = os.path.join(CKPT_DIR, f"step{step + 1}")
        os.makedirs(ckpt_path, exist_ok=True)

        student.save_pretrained(ckpt_path)
        tokenizer.save_pretrained(ckpt_path)
        torch.save(optimizer.state_dict(), os.path.join(ckpt_path, "optimizer.pt"))
        torch.save(
            {"cpu": torch.get_rng_state(),
             "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None},
            os.path.join(ckpt_path, "rng.pt"),
        )
        if os.path.exists(LOG_FILE):
            shutil.copy(LOG_FILE, os.path.join(ckpt_path, "training_log.jsonl"))

        state = {
            "step": step + 1, "best_val_score": float(best_val_score),
            "best_step": int(best_step), "wandb_run_id": wandb_run_id,
            "base_val_score": base_val_score,
        }
        with open(os.path.join(ckpt_path, "training_state.json"), "w") as f:
            json.dump(state, f)
        print(f"  Saved checkpoint: {ckpt_path}")

        # Push to Hub (only if checkpointing)
        if PUSH_TO_HUB and HF_TOKEN:
            api = HfApi()
            api.create_repo(HF_HUB_ID, exist_ok=True, token=HF_TOKEN)
            api.upload_folder(
                folder_path=ckpt_path,
                repo_id=HF_HUB_ID,
                path_in_repo=f"step{step + 1}", token=HF_TOKEN,
                commit_message=f"Step {step + 1} (DistilledRL)",
            )
            print(f"  Pushed to HF Hub: {HF_HUB_ID}/step{step + 1}")
    # ========================= VALIDATION =========================
    if (step + 1) % VAL_EVERY == 0:
        val_score, val_preds = run_validation(student, val_ds)  # FIXED: unpacks correctly
        delta = val_score - (base_val_score or 0.0)
        print(f"  >>> Validation Pass@1: {val_score:.4f} (base: {base_val_score}, delta: {delta:+.4f})")
        if USE_WANDB:
            wandb.log({"val/pass_at_1": val_score, "val/delta_vs_base": delta}, step=step)

        # Save predictions
        val_pred_file = os.path.join(CKPT_DIR, f"val_preds_step{step+1}.jsonl")
        with open(val_pred_file, "w") as f:
            for p in val_preds:
                f.write(json.dumps(p) + "\n")

        if val_score > best_val_score:
            best_val_score = val_score
            best_step = step + 1
            os.makedirs(BEST_DIR, exist_ok=True)
            student.save_pretrained(BEST_DIR)
            tokenizer.save_pretrained(BEST_DIR)
            print(f"  *** New best validation score! Saved to {BEST_DIR}")
            if USE_WANDB:
                wandb.run.summary["best_val_pass@1"] = best_val_score
                wandb.run.summary["best_step"] = best_step

# ========================= FINAL =========================
print("\nTraining complete. Running final validation...")
final_val, _ = run_validation(student, val_ds)  # FIXED: no unpacking error
print(f"Base student Pass@1:   {base_val_score}")
print(f"Final validation Pass@1: {final_val:.4f}")
print(f"Best validation Pass@1:  {best_val_score:.4f} @ step {best_step}")
print(f"Dead steps skipped: {n_dead_steps}")

if USE_WANDB:
    wandb.log({"val/final_pass_at_1": final_val}, step=MAX_STEPS)
    wandb.run.summary["final_val_pass@1"] = final_val
    wandb.run.summary["dead_steps"] = n_dead_steps
    wandb.finish()

final_path = os.path.join(CKPT_DIR, "final")
student.save_pretrained(final_path)
tokenizer.save_pretrained(final_path)
print(f"Final checkpoint: {final_path}")